# BlackSwan: Embedding + CNN for Extreme Event Semantic Analysis
### An Advanced Research Workbook on Exogenous Shock Latency & Information Microstructure
*Pedagogical alignment: Embedding + 1D CNN for Text Classification with Pre-Trained Transformers*

**Student:** Gwendalynn Lim Wan Ting  
**Institution:** Singapore Institute of Technology (SIT)  
**Student Email:** `2504137@sit.singaporetech.edu.sg`  
**Program:** Information and Communications Technology / Applied Computing (AI & ML)  
**Course Module:** ICT3506C Applied Natural Language Processing (Lab 2.4 Research Extension)  
**AI Collaboration:** Pair-programmed with **Google Antigravity** (Google DeepMind)

---

## 🔬 Lab Context & Introduction
Conventional natural language processing (NLP) tutorials focus on retail sentiment analysis (classifying financial headlines into positive, neutral, or negative classes). While foundational, real-world quantitative trading and risk desks rarely use static headline sentiment for trade execution because high-frequency order flows price news in sub-millisecond intervals.

Instead, quantitative research desks frequently utilize **Convolutional Neural Networks (CNNs)** coupled with **pre-trained Transformer embeddings** for **exogenous shock forensics**:
1. Analyzing how quickly an ambiguous black-swan event mutates from localized uncertainty (*"unconfirmed report"*, *"minor disruption"*) to confirmed systemic market shock (*"airspace closure"*, *"coordinated attack"*).
2. Measuring the semantic latency between the physical onset of an event and the time the financial ecosystem mathematically reflects that risk.

In this workbook, you will implement a **Multi-Scale 1D CNN** using pre-trained **BERT embeddings** as transfer learning inputs, while preserving the full architectural rubric of the syllabus.


## 🎯 Core Curricular Objectives
By completing this workbook, you will:
1. **Data Preparation**: Process timestamped news headlines into token index matrices and attention masks.
2. **Transfer Learning**: Leverage pre-trained BERT embeddings (`bert-base-uncased`) as the dense input representation for the convolutional network.
3. **Model Building**: Construct a 1D Convolutional Neural Network with multiple parallel kernel sizes ($k \in \{2, 3, 4\}$) to capture key $n$-gram phrases, followed by global max-over-time pooling.
4. **Training & Evaluation**: Train the network and evaluate its classification metrics (Precision, Recall, F1-Score).
5. **Latency Forensics**: Track and plot the temporal activation of shock-detection kernels across a real-time event window.


---
## Part 1: Environment Setup & Dependencies
Import the necessary deep learning libraries (PyTorch, Hugging Face Transformers, Scikit-Learn, Pandas, Matplotlib).


In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from transformers import BertTokenizerFast, BertModel

# Device selection (CUDA / MPS / CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on device: {device}")


---
## Part 2: Data Preparation & Micro-Temporal Structuring
In this section, we load the event sequence dataset. 

Each sample contains:
- `timestamp`: Time of report
- `headline`: Raw wire text
- `ground_truth_class`: Category index (0: Uncertainty/Localized, 1: Early Impact, 2: Systemic Shock)

Your task: Complete the `EventStreamDataset` class below.


In [ ]:
class EventStreamDataset(Dataset):
    """
    PyTorch Dataset for timestamped news headlines.
    """
    def __init__(self, data_path, tokenizer, max_len=64):
        with open(data_path, 'r', encoding='utf-8') as f:
            self.records = json.load(f)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.records)
        
    def __getitem__(self, idx):
        item = self.records[idx]
        text = item['headline']
        label = item['ground_truth_class']
        
        # TODO: Tokenize text using self.tokenizer
        # Required parameters:
        # - max_length = self.max_len
        # - padding = 'max_length'
        # - truncation = True
        # - return_tensors = 'pt'
        
        # --- YOUR CODE HERE ---
        
        # Expected outputs:
        # return {
        #     'input_ids': encoding['input_ids'].squeeze(0),
        #     'attention_mask': encoding['attention_mask'].squeeze(0),
        #     'label': torch.tensor(label, dtype=torch.long),
        #     'timestamp': item['timestamp']
        # }
        pass


---
## Part 3: Transfer Learning via Pre-trained BERT Embeddings
Rather than training word vectors from scratch on a small corpus, we leverage pre-trained Transformer representations (`bert-base-uncased`).

### Architectural Choice: Frozen vs Fine-Tuned
In this lab, we freeze the base BERT weights so the model acts purely as a deterministic feature extractor, drastically reducing training time while ensuring our 1D CNN heads learn localized semantic triggers.

$$E = \text{BERT}_{\text{frozen}}(\mathbf{x}) \in \mathbb{R}^{B \times L \times 768}$$


In [ ]:
# Initialize pre-trained tokenizer and model
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
bert_backbone = BertModel.from_pretrained('bert-base-uncased')

# TODO: Freeze all parameters in bert_backbone
# for param in bert_backbone.parameters():
#     param.requires_grad = False

# Verify parameter freezing
trainable_params = sum(p.numel() for p in bert_backbone.parameters() if p.requires_grad)
print(f"Trainable parameters in BERT backbone: {trainable_params} (Expected: 0)")


---
## Part 4: Constructing the Multi-Scale 1D CNN
Standard sentiment analysis relies on fixed bag-of-words or single-kernel CNNs. For event forensics, key information is captured at multiple linguistic scales:
- **$k=2$ (Bigrams):** Catches rapid associations (*"small plane"*, *"delayed opening"*)
- **$k=3$ (Trigrams):** Catches causal clauses (*"struck south tower"*, *"halts all departures"*)
- **$k=4$ (4-grams):** Catches systemic institutional directives (*"delay opening bell indefinitely"*)

### Architecture:
1. **Input Dimension:** Shape $(B, 768, L)$ (transposed from BERT's $(B, L, 768)$).
2. **Convolutional Bank:** Parallel 1D convolutions with filter sizes $k \in \{2, 3, 4\}$, each with $C_{\text{out}} = 128$.
3. **Activation:** ReLU or GELU non-linearity.
4. **Max-over-Time Pooling:** Global max pooling across the temporal sequence dimension $L - k + 1$.
5. **Concatenation & Classification:** Concatenate all pooled vectors into dimension $3 \times 128 = 384$, apply Dropout ($p=0.3$), and project to class logits via a linear layer.


In [ ]:
class MultiScaleCNNTextClassifier(nn.Module):
    def __init__(self, bert_model, num_classes=3, num_filters=128, filter_sizes=(2, 3, 4), dropout=0.3):
        super(MultiScaleCNNTextClassifier, self).__init__()
        self.bert = bert_model
        embedding_dim = bert_model.config.hidden_size # 768 for bert-base
        
        # TODO: Define a bank of 1D convolutions using nn.ModuleList
        # Each convolution should map:
        # in_channels = embedding_dim (768)
        # out_channels = num_filters (128)
        # kernel_size = k (from filter_sizes)
        
        # --- YOUR CODE HERE ---
        # self.convs = nn.ModuleList([
        #     nn.Conv1d(in_channels=embedding_dim, out_channels=num_filters, kernel_size=k)
        #     for k in filter_sizes
        # ])
        
        # self.dropout = nn.Dropout(dropout)
        # self.fc = nn.Linear(len(filter_sizes) * num_filters, num_classes)
        pass

    def forward(self, input_ids, attention_mask):
        # 1. Pass tokens through frozen BERT to extract last hidden states
        # with torch.no_grad():
        #     bert_outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        #     embeddings = bert_outputs.last_hidden_state  # Shape: (Batch, Seq_Len, 768)
            
        # 2. Transpose for Conv1D: PyTorch Conv1d expects (Batch, Channels, Seq_Len)
        # x = embeddings.permute(0, 2, 1)
        
        # 3. Apply convolutions, activations, and max-over-time pooling
        # conv_results = []
        # for conv in self.convs:
        #     c = F.relu(conv(x))                         # Shape: (Batch, num_filters, L - k + 1)
        #     pooled = F.max_pool1d(c, kernel_size=c.shape[2]).squeeze(2) # Shape: (Batch, num_filters)
        #     conv_results.append(pooled)
            
        # 4. Concatenate feature maps
        # z = torch.cat(conv_results, dim=1)              # Shape: (Batch, len(filter_sizes) * num_filters)
        
        # 5. Regularization and Dense classification
        # logits = self.fc(self.dropout(z))               # Shape: (Batch, num_classes)
        # return logits
        pass


---
## Part 5: Training Loop & Rubric Evaluation
In this section, we set up the training loop using cross-entropy loss and the AdamW optimizer.


In [ ]:
# Hyperparameters
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
EPOCHS = 5

# TODO: Instantiate Dataset, DataLoader, Model, Criterion, and Optimizer
# dataset = EventStreamDataset('../data/sample_event_stream.json', tokenizer)
# dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
# model = MultiScaleCNNTextClassifier(bert_backbone, num_classes=3).to(device)
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        # TODO: Implement standard PyTorch forward-backward step
        pass
    return total_loss


---
## Part 6: Latency Analysis & Activation Topography
Once the CNN is trained, we pass the chronological sequence of headlines through the model to track the **semantic phase-shift** from localized uncertainty to systemic crisis.

This produces the forensic dual-axis chart:
- **X-axis:** Timestamp (8:46 AM to 9:15 AM)
- **Y-axis:** Class probability distribution ($P(\text{Shock})$ vs $P(\text{Uncertainty})$)


In [ ]:
def run_forensic_latency_analysis(model, dataset):
    """
    Computes chronological shock probability across the event stream.
    """
    model.eval()
    timestamps = []
    shock_probs = []
    
    with torch.no_grad():
        for i in range(len(dataset)):
            sample = dataset[i]
            input_ids = sample['input_ids'].unsqueeze(0).to(device)
            attention_mask = sample['attention_mask'].unsqueeze(0).to(device)
            
            # TODO: Get model prediction logits and apply Softmax
            # logits = model(input_ids, attention_mask)
            # probs = F.softmax(logits, dim=1)
            # shock_prob = probs[0, 2].item() # Class 2 = SYSTEMIC_SHOCK
            
            timestamps.append(sample['timestamp'])
            # shock_probs.append(shock_prob)
            
    # TODO: Plot the time-series activation curve
    # plt.figure(figsize=(10, 5))
    # plt.plot(timestamps, shock_probs, marker='o', color='crimson', label='P(Systemic Shock)')
    # plt.title('Exogenous Shock Latency: Semantic Mutation over Time')
    # plt.xlabel('Timestamp (ET)')
    # plt.ylabel('Model Confidence')
    # plt.grid(True)
    # plt.show()
    pass


---
## Part 7: Quantitative Defense & Discussion Questions

1. **Information Asymmetry & HFT:**  
   *Explain why a 15ms inference latency on a 1D CNN is sufficient for macroeconomic risk rebalancing, but completely obsolete for collocated High-Frequency Trading (HFT) order-matching.*

2. **Kernel Filter Size Interpretation:**  
   *Why does a 4-gram filter ($k=4$) capture structural market events with higher specificity than a bigram filter ($k=2$), and what is the trade-off regarding sample sparsity?*

3. **Temporal Invariance:**  
   *What role does Max-over-Time Pooling play in ensuring that the position of a shock keyword (beginning vs. end of headline) does not degrade model confidence?*
